FEATURE ENGINEERING E GESTIONE DATASET

Dopo aver caricato i dati ed averli puliti, è arrivato il momento di fare un passo avanti.
Passiamo dalla materia grezza, alla materia lavorata.
Trasformiamo i dati in segnali che il modello può digerire facilmente.
Gestiamo la trasformazione ma anche la strategia per evitare che il modello imbrogli durante l'addestramento.

- Preprocessing pipeline: creare pipeline scikit-learn serializzabili
- Gestione sbilanciamento: oversampling, smote e class weights. come insegnare all'ai a riconoscere un evento raro (come una froda bancaria)
- Data splitting: per evitare data leakage e time-serie split

FEATURE ENGINEERING

Il Feature Enginnering costruisce le informazioni che dai al modello. La gestione del dataset garantisce che quelle informazioni siano corrette, tracciabili e utilizzate senza falsare la valutazione.

I confini tra Preprocessing e Feature Engineering sono spesso sovrapposti, non sono rigidamente separati.
Preprocessing: correggi tipi, tratti informazioni mancanti, codifichi categorie e scali i valori
Feature Engineering: costruizione di rappresentazioni informative, come il consumo precedente o tempo dall'ultimo movimento di magazzino.

L'errore più grave è utilizzare il futuro
Usare informazioni indisponibili al momento della previsione produce data leakage: risultati di test troppo ottimistici.
Anche imputazione, standarizzazione e selezione delle feature devono apprendere i propri parametri solo dal training, o dalla parte di training di ciascuna validazione. Le pipeline aiutano a rispettare questa separazione.
Nella gestione conta anche quando l'informazione è stata registrata. Un movimento con data 31/03, inserito il 05/04 non era disponibile al 31/03 giorno delle nostre analisi. Questa è una conseguenza pratica della regola sulla disponibilità temporale.

GESTIONE DEL DATASET

Si intende le attività necessarie a organizzare, documentare, controllare e conservare i dati lungo il progetto.

Un salvataggio dei file potrebbe essere:
- nomefile_Raw: estrazione originale, conservata senza correzioni manuali
- nomefile_Processed: dati puliti e uniformati
- nomefile_Features: variabili costruite e target per l'addestramento
- nomefile_Split: definizione delle righe assegnate a training, validazione e test.
Il file .parquet può contenere qualsiasi di questi livelli. Parquet è il formato del file, non stabilisce se il dataset è corretto o pronto per il modello
DVC può tracciare le versione dei file e collegarle al codice tramite Git. Non verifica da solo il significato aziendale dei dati.

Il dataset originale va diviso tra Training (addestra il modello), Validazione (per confrontare modelli e scegliere i parametri), Test (per la valutazione finale).
Il test va tenuto separato dalle scelte di sviluppo. Una divisione casuale delle righe può dare una valutazione troppo favorevole nelle serie temporali, è preferibile simulare previsioni dal passato verso il futuro, eventualmente con più finestre temporali.
Per calcolare una feature di gennaio 2026 puoi utilizzare dicembre 2025, quel passato serebbe disponibile anche nell'utilizzo reale. Non puoi invece usare febbraio 2026.


Come garantire che il modella riceva sempre dati coerenti?

Preprocessing e Pipeline Professionali
Garantire coerenza tra training e inferenza.
Uno dei problemi più comuni nel deployment è l'incoerenza dei dati. Se il modello è addestarto su dati scalati in un modo, ma riceve input grezzi in produzione, le predizioni saranno errate.
E' come montare un mobile Ikea con istruzioni sbagliate, il modello si stente così se riceve dati diversi rispetto a quelli per cui è stato addestrato.
La soluzione risiede nell'uso di PIPELINE PROFESSIONAL serializzabili che incapsulano ogni trasformazione, dal 'missing value imputation' alla normalizzazione finale.

Quali sono le fasi della Pipeline?
Automazione del Preprocessing
L'automazione si base sul concetto di pipeline object si scikit learn.
Immaginalo come un nastro trasportatore, il dato entra grezzo e passa attraverso stazioni di trasformazione. 
Incapsulare la logica di trasformazione
- Pipeline Object: un contenitore che sequenzializza i passaggi di 'fit' e 'transform' in un unico oggetto Python
- Serialization: utilizzo di librerie come 'joblib' o 'pickle' per salvare la pipeline e ricaricarla nel server di produzione. Salvo l'intero nastro trasportatore e lo spedisco al server di produzione.
- ColumnTransformer: gestione differenziata per colonne numeriche (scaling) e categoriche (encoding) all'interno dello stesso flusso
L'output della pipeline garantisce che ogni input 'x' venga trasformato secondo i parametri appresi dal set di addestramento.

Componenti della Pipeline
i componenti fondamentali sono 3:
1) StandardScaler: trasforma le caratteristiche numeriche affinchè abbiano media zero e varianza unitaria, fondamentale per la convergenza delle reti neurali. Senza di esso le rete neurali faticano a convergere. Perchè alcune caretteristiche sono più forti di altre solo perchè hanno numeri più grandi.
2) OneHotEncoder: convertire variabili categoriche in vettori binari 'sparse' evitando di introdurre ordini numerici fittizi tra le categorie non correlate. Non vogliamo che la categoria 3 sia superiore a 1 se stiamo parlando di colori.
3) Custom Transformers: creazione di classi personalizzate che ereditano da 'BaseEstimator' per applicare logiche di business specifiche (es. estrazione di feature da timestamp). Questo se i dati non bastano, possiamo aggiungere colonne per aiutare il modello. Aggiugnere coscenza umana ai bit grezzi.

Ma cosa succede se dimentichiamo di utilizzare questi strumenti in modo integrato?

Training-Serving Skew
Evitare divergenze pericolose.
Il 'skew' si verifica quando la distribuzione dei dati di produzione differisce sistematicamente da quella di training a causa di trasformazioni manuali asincrone.
Il preprocessing fatto a mano durante la ricerca, differisce da quello implementato dal programmaotre nel back-end.
L'uso di pipeline integrate riduce il rischio di errori umani, garantendo che lo stato delle trasformazioni sia immutabile dopo il training. Trasformiamo il preprocessing in una fase atomica, o applica tutto lo strato del training o non applica nulla.

Gestione dello Sbilanciamento delle Classi
Insegnare al modello a riconoscere le rarità
In molti casi reali, come il rilevamento frodi o diagnosi mediche, la classe di interesse rappresenta meno dell'1% dei dati totali.
E' necessario bilanciare il dataset senza perdere informazioni cruciali, agendo sia sui dati che sulla funzione di perdita (loss function).
Se vuoi istruire un guardiano a riconoscere i ladri in un aeroporto, dovo passano 1 milioni di persone oneste e solo 10 ladri. se il modello predisce sempre 'onesto' ha ragione nel 99.9% dei casi, ma ha fallito il suo compito
Lo sbilanciamento delle classi rende l'accuratezza una metrica bugiarda. Dobbiuamo forzare il modello a dare importanza alle rarità, agendo sulla sua punizione quando sbagli sui casi difficili.

Abbiamo 3 tecniche per riequilibrare questo sbilanciamento
1) Random Oversampling: duplicazione causale dei campioni della classe minoritaria per aumentare il peso statistico. copia i dati rari ma rischi overfitting
2) SMOTE (Synthetic Minority Over-sampling Technique): creazione di nuovi campioni sintetici tramite interpolazione lineare tra vicini. 
3) Class Weights: modifica della funzione di costo (la loss) per penalizzare maggiormente gli errori sulla classe rara. E' come dare al modello una scossa elettrica più forte quando ignora la rarità
La funzione di perdita pesata assegna un'importanza aggiore ai campioni appartenenti a classi con minore frequenza

Per capire se queste tecniche funzionano dobbiamo cambiare il modo con cui misuriamo il successo.

Valutazione dello Sbilanciamento
Dimenticare l'accuratezza, guardiamo la confusion matrix. Dobbiamo monitorare la Precision (quanti dei sospettati sono davvero colpevoli) e la Recall Curve (quanti dei colpevoli dobbiamo ancora catturare).
La F1-score è la loro media armonica. Media armonica tra precisione e richiamo, utile per avere un unico indicatore di performance bilanciata.
Con la Cunfusion Matrix facciamo analisi visiva dei falsi positivi e falsi negativi per capire dove il modello sta fallend sistematicamente.
L'obbietivo è l'equilibrio

Approfondiamo come SMOTE crea i sui campioni statistici

Interpretazione Geometrice di SMOTE
Oltre la semplice duplicazione
Smote non copia i dati, me esplora lo spazio delle caratteristiche tra i campioni esistenti, riducendo il rischio di overtiffing.
Il nuovo campione 's' (dato da smote) viene generato prendendo un campione 'x' della classe minoritario e cerca i suo vicini, poi traccia una linea e posiziona un nuovo dato sinteticamente in un punto casuale 'u' su quella linea.
Espandiamo la conoscenza del modello per interpolazione.

Integrita temporale dei dati
Dataset Splitting e Data Leakage
Il 'Data Leakage' si verifica quando informazioni del set di test 'scivolano' in quello di trainig, portando a performance artificialmetne alte e fallimenti in produzione.
Per evitare il data leakage i dati vanno divisi correttamente, specialmente quando esiste una componente temporale (time-series).

Come possiamo dividere i dati senza inquinare il test set.
Dobbiamo scegliere lo split corretto
Isolare correttamente i dati
- Stratified Split: mantiene le proporzioni tra le classi. in questo modo, le proporzioni tra le classi è la stessa tra training, validation e test set
- Time-series Split: evita di usare dati del 'futuro' per predire il 'passato' mantenendo l'ordine cronologico. Addestriamo sul passato, testiamo sul presente, validiamo sul futuro. Mai mescolare i tempi.
- Group Split: assicura che campioni correlati (es. foto diverse dello stesso utente) non vengano divisi tra train e test.
Il leakage invalida la stima dell'errore di generalizzazione, rendendo il modello inaffidabile.

Dove si nascondono solitamenta questi Leakage?
- Scaling Leakage: avviene quando calcoli la media dell'intero dataset prima di dividerlo tra training e set. In questo modo hai appena detto al training set quel'è la media del futuro.
- Temporale Leakage: includere fueature che sono state generate 'dopo' l'evento che si vuole predire (es. usare il prezzo di chiusura per predire il trend della giornata)
- Target Leakage: utilizzare colonne che sono correlate quasi perfettamente con il target ma che non saranno disponibili al momento dell'inferenza reale. Se il modello impara a leggere la colonna che è quasi un sinonimo del target, non sta imparando a fare ai, sta solo leggendo una etichetta nascosta.

Come validare questi modelli in modo dinamico.
Cross-Validation Temporale.
Validare serie storiche
Nelle serie temporali non possiamo usare la K-Fold standard (perchè mescolerebbe i giorni) poichè i dati non sono indipendenti e identicamente distribuiti
Dobbiamo utilizzare una finestra mobile o espansiva che rispetti la freccia del tempo per ogni fold di validazione.
Questo garantisce che il modello impari a gestire la dinamicità e l'evoluzione dei dati.




In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

# --- 1. GENERAZIONE DEL DATASET SINTETICO ---
# Creiamo un dataset di 1000 campioni che simula il rischio di insolvenza (default) dei prestiti.
n_samples = 1000
np.random.seed(42)

# Feature 1: Stipendio Annuo. Utilizziamo una distribuzione Gamma per renderla più realistica (molti medi, pochi alti).
stipendio = np.random.gamma(shape=5, scale=8000, size=n_samples) 
# Feature 2: Tipo di Contratto. Variabile categorica con frequenze predefinite.
tipo_contratto = np.random.choice(['indeterminato', 'determinato', 'part-time'], 
                                  size=n_samples, p=[0.6, 0.25, 0.15])

# Definizione del target (rischio_default) basato su logica statistica:
# Il rischio diminuisce all'aumentare dello stipendio e aumenta con contratti meno stabili.
rischio_score = -0.6 * (stipendio / 10000)
rischio_score += np.where(tipo_contratto == 'determinato', 1.0, 0)
rischio_score += np.where(tipo_contratto == 'part-time', 2.0, 0)
rischio_score += np.random.normal(0, 1, size=n_samples) # Aggiunta di rumore per non renderlo deterministico

# Creiamo uno sbilanciamento forzato: solo il top 10% dei punteggi di rischio viene marcato come '1' (Default).
soglia = np.percentile(rischio_score, 90)
rischio_default = (rischio_score >= soglia).astype(int)

# Organizzazione dei dati in un DataFrame Pandas per una gestione più semplice
df = pd.DataFrame({
    'stipendio_annuo': stipendio,
    'tipo_contratto': tipo_contratto,
    'rischio_default': rischio_default
})

# --- 2. SUDDIVISIONE DEI DATI (STRATEGIA ANTI-LEAKAGE) ---
# Separazione tra variabili indipendenti (X) e target (y).
X = df.drop('rischio_default', axis=1)
y = df['rischio_default']

# Suddivisione in Train e Test (80/20).
# Usiamo 'stratify=y' per garantire che la proporzione di 10% di default sia mantenuta in entrambi i set.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- 3. PIPELINE DI PRE-PROCESSING ---
# Poiché abbiamo tipi di dati diversi (numerici e categorici), dobbiamo applicare trasformazioni mirate.
numeric_features = ['stipendio_annuo']
categorical_features = ['tipo_contratto']

# ColumnTransformer permette di applicare trasformazioni diverse a colonne diverse in un unico passaggio.
preprocessor = ColumnTransformer(
    transformers=[
        # StandardScaler: Centra la media a 0 e la deviazione standard a 1 (fondamentale per reti neurali).
        ('num', StandardScaler(), numeric_features),
        # OneHotEncoder: Converte stringhe in vettori binari (es. 'part-time' -> [0, 0, 1]).
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# FIT dello scaler solo sui dati di TRAIN per evitare Data Leakage.
X_train_proc = preprocessor.fit_transform(X_train)

# TRASFORMAZIONE del TEST set usando SOLO le statistiche apprese dal train.
X_test_proc = preprocessor.transform(X_test)

# Conversione dei risultati (che sono array NumPy) in Tensori PyTorch.
X_train_t = torch.tensor(X_train_proc, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

# --- 4. CALCOLO DEI PESI PER LE CLASSI SBILANCIATE ---
# Calcoliamo i pesi correttivi per compensare la rarità della classe 1 (guasti/default).
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
# In BCELoss, il peso 'pos_weight' si riferisce specificamente alla classe 1.
# weights[1] contiene il peso calcolato per la classe 1.
class_weights = torch.tensor(weights[1], dtype=torch.float32)
print(f"Peso correttivo assegnato alla classe rara (1): {class_weights:.2f}")

# --- 5. DEFINIZIONE DELL'ARCHITETTURA DEL MODELLO ---
class SimpleClassifier(nn.Module):
    """
    Rete neurale semplice con un layer nascosto.
    Interazioni:
    - __init__: Definisce gli strati della rete (Linear -> ReLU -> Linear).
    - forward: Definisce come i dati fluiscono attraverso la rete.
    """
    def __init__(self, input_shape):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_shape, 4), # Primo strato lineare
            nn.ReLU(),                 # Funzione di attivazione non lineare
            nn.Linear(4, 1)            # Layer di output (senza Sigmoid, gestita dalla Loss)
        )
    
    def forward(self, x):
        return self.net(x)

# Inizializzazione del modello basandosi sul numero di feature dopo l'encoding
model = SimpleClassifier(X_train_proc.shape[1])

# --- 6. CONFIGURAZIONE TRAINING ---
# BCEWithLogitsLoss è ideale per classificazioni binarie.
# Passando 'pos_weight', diciamo al modello che sbagliare sulla classe 1 è "più grave" del peso fornito.
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)

# Ottimizzatore Adam con learning rate 0.01 per aggiornare i pesi.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Ciclo di addestramento rapido
for epoch in range(100):
    optimizer.zero_grad()      # Reset dei gradienti
    outputs = model(X_train_t) # Forward pass
    loss = criterion(outputs, y_train_t) # Calcolo della loss pesata
    loss.backward()            # Calcolo dei gradienti (Backprop)
    optimizer.step()           # Aggiornamento pesi

print(f"Addestramento completato con successo. Loss finale: {loss.item():.4f}")